In [1]:
from helpers.models import Models
from helpers.llm_client import LLMClient
from helpers.functions import *
from helpers.parsers import license_parser
import pandas as pd
import os
import nirjas
from sklearn.metrics import accuracy_score
import re

pd.set_option('display.max_rows', None)    # Show all rows
# pd.set_option('display.max_colwidth', None)  # Show full column width

/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
# Check to make sure that all API keys are present
os.environ['GROQ_API_KEY'] 
os.environ['NVIDIA_API_KEY']
os.environ['TOGETHER_API_KEY']    
'OK'
#

'OK'

In [3]:
test_file_path = 'extras/LastGoodNomosTestfilesScan.txt'
test_file_columns = ['file path', 'licenses']

with open(test_file_path, 'r') as file:
    test_data = file.readlines()

test_df = pd.DataFrame(columns=test_file_columns)

for line in test_data:
    if "contains license(s)" in line:  # Check for the expected pattern
        licenses = line.split("contains license(s) ")[1].strip().split("\\")[0]
        file_path = line.split('File ')[1].split(' contains license(s)')[0].strip()
        licenses = str(licenses)
        licenses = licenses.split(',')
        licenses = '\n'.join(licenses)
        temp_df = pd.DataFrame({'file path': file_path, 'licenses': [licenses]})
        test_df = pd.concat([test_df, temp_df], ignore_index=True)

test_df.loc[1997, 'licenses']

'Dual-license\nGPL\nToolbar2000'

In [4]:
for index, row in test_df.iterrows():
    try:
        with open(os.path.join('extras', row['file path']), "r", encoding='utf-8') as f:
            comments = f.read()
    except:
        print(f'Dropping Index: {index}')
        test_df = test_df.drop(index)
test_df = test_df.reset_index()

Dropping Index: 39
Dropping Index: 83
Dropping Index: 160
Dropping Index: 161
Dropping Index: 162
Dropping Index: 202
Dropping Index: 301
Dropping Index: 322
Dropping Index: 350
Dropping Index: 548
Dropping Index: 625
Dropping Index: 646
Dropping Index: 775
Dropping Index: 888
Dropping Index: 902
Dropping Index: 916
Dropping Index: 939
Dropping Index: 940
Dropping Index: 963
Dropping Index: 1394
Dropping Index: 1398
Dropping Index: 1399
Dropping Index: 1439
Dropping Index: 2009


In [5]:
len(test_df)

2054

In [6]:
create_license_dataset('extras/license_information/details')
client = LLMClient()
model = SentenceTransformer("all-mpnet-base-v2")
test_df = extract_comments(test_df)

License dataset file created successfully at extras/license_information/license_dataset.csv


/home/jimbo/Desktop/GSoC24/repo/GSoC24/gsoc24env/lib/python3.8/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
test_df.loc[40]
# print(test_df.loc[40, 'file_comments'])

index                                                           41
file path                     NomosTestfiles/Apache/Apache-1.0.txt
licenses                                                Apache-1.0
file_comments    Copyright (c) 1995-1999 The Apache Group.  All...
Name: 40, dtype: object

In [8]:
print(test_df.loc[40, 'file_comments'])

Copyright (c) 1995-1999 The Apache Group.  All rights reserved.
Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:
1. Redistributions of source code must retain the above copyright notice, this list of conditions and the following disclaimer.
2. Redistributions in binary form must reproduce the above copyright notice, this list of conditions and the following disclaimer in the documentation and/or other materials provided with the distribution.
3. All advertising materials mentioning features or use of this software must display the following acknowledgment:"This product includes software developed by the Apache Group for use in the Apache HTTP server project (http://www.apache.org/)."
4. The names "Apache Server" and "Apache Group" must not be used to endorse or promote products derived from this software without prior written permission. For written permission, please contact apache@apache.org

In [9]:
lines_tuple = get_top_similar_license_lines(\
        code_text = test_df.loc[40, 'file_comments'],
        licenses_file_path = 'extras/license_information/license_dataset.csv',
        model = model,
        top_k = 10,
        min_similarity = 60,
        double_semantic_search = True
    )
lines_tuple

[(99.0,
  'Copyright (c) 1995-1999 The Apache Group.  All rights reserved.',
  'Apache License 1.0',
  'Apache-1.0',
  'Copyright (c) 1995-1999 The Apache Group. All rights reserved.',
  [('Apache License 1.0', 99.0),
   ('Apache License 1.0', 99.0),
   ('PHP License v3.01', 84.0),
   ('PHP License v3.01', 84.0),
   ('PHP License v3.0', 82.0)]),
 (100.0,
  'Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:',
  'Mackerras 3-Clause - acknowledgment variant',
  'Mackerras-3-Clause-acknowledgment',
  ' Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met:',
  [('Net-SNMP License', 100.0),
   ('BSD with attribution', 100.0),
   ('TMate Open Source License', 100.0),
   ('TMate Open Source License', 100.0),
   ('Mup License', 100.0)]),
 (100.0,
  '1. Redistributions of source code must retain the above copyright n

In [22]:
for index, row in test_df.loc[0:999].iterrows():
    lines_tuple = get_top_similar_license_lines(\
                    code_text = row['file_comments'],
                    licenses_file_path = 'extras/license_information/license_dataset.csv',
                    model = model,
                    top_k = 10,
                    double_semantic_search=True,
                    min_similarity=60
            )
    only_relevant_lines = []
    relevant_lines_with_predictions = []
    for tup in lines_tuple:
        only_relevant_lines.append(tup[1])
        relevant_lines_with_predictions.append((tup[1], tup[3]))
    test_df.loc[index, 'tuples_all_info'] = str(lines_tuple)
    test_df.loc[index, 'tuples_no_sim_score_no_predictions'] = str(only_relevant_lines)
    test_df.loc[index, 'tuples_no_sim_score_with_predictions'] = str(relevant_lines_with_predictions)
    lines = [line[1] for line in lines_tuple]
    lines = '\n'.join(lines)
    test_df.loc[index, 'license_relevant_lines'] = lines
    lines = [line[2] for line in lines_tuple]
    lines = '\n'.join(lines)
    test_df.loc[index, 'predicted_license_names'] = lines
    lines = [line[3] for line in lines_tuple]
    lines = '\n'.join(lines)
    test_df.loc[index, 'predicted_license_ids'] = lines
test_df.to_csv('predicted_nomos_results.csv')
for index, row in test_df[0:1000].iterrows():
    test_df.loc[index, 'predicted_license_accurate'] = predicted_license_found(row['predicted_license_ids'], row['licenses'])
    test_df.loc[index, 'predicted_licenses_covered'] = predicted_license_covered(row['predicted_license_ids'], row['licenses'])
print('Predicted License Accuracy:', accuracy_score([1] * 1000, test_df.loc[0:999]['predicted_license_accurate']))
print('Predicted Licenses Covered:', np.average(test_df.loc[0:999]['predicted_licenses_covered']))

Predicted License Accuracy: 0.651
Predicted Licenses Covered: 58.2


In [6]:
test_df.loc[25]

Unnamed: 0                                                                             25
index                                                                                  25
file path                                          NomosTestfiles/AGPL/AGPL-3.0_ref_d.txt
licenses                                                                    AGPL-3.0-only
file_comments                           # rpm-specific\nset(CPACK_RPM_PACKAGE_SUMMARY ...
tuples                                                                                NaN
license_relevant_lines                                                                NaN
predicted_license_names                                                               NaN
predicted_license_ids                                                                 NaN
predicted_license_accurate                                                            NaN
predicted_licenses_covered                                                            NaN
tuples_all

In [4]:
test_df = pd.read_csv('predicted_nomos_results.csv')

In [5]:
miss_classifcations = test_df.loc[0:1000][test_df['predicted_license_accurate'] != 1]
miss_classifcations[['tuples_no_sim_score_no_predictions','predicted_license_ids','licenses',]]

/tmp/ipykernel_53619/2590909624.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  miss_classifcations = test_df.loc[0:1000][test_df['predicted_license_accurate'] != 1]


,tuples_no_sim_score_no_predictions,predicted_license_ids,licenses
3,"['-- SOFTWARE, DOCUMENTATION OR OTHER INFO...",CC0-1.0\nHPND-export2-US\nPixar,Govt-rights\nUnclassifiedLicense
4,[''],CNRI-Jython,ACE
7,"['Permission to use, copy, modify, distribute,...",Adobe-Display-PostScript\nAdobe-Display-PostSc...,MIT-style
9,['9. GENERALThis Agreement supersedes any prio...,Nokia\nEPL-2.0\nEPL-2.0\nHPND-export2-US\nSSPL...,Adobe-DNG
10,['Adobe shall not be liable to any party for a...,Adobe-Glyph\nAdobe-Glyph\nAdobe-Glyph\nAdobe-G...,Adobe-Glyph
11,"['THE INFORMATION BELOW IS FURNISHED AS IS, IS...",Afmparse\nAfmparse\nAfmparse\nAfmparse\nAfmpar...,Afmparse
12,"['License to Source Code. The term ""Source Cod...",AFL-1.2\nAFL-1.2\nAFL-1.2\nAFL-1.1\nCC-BY-NC-S...,AFL-1.1
13,"['License to Source Code. The term ""Source Cod...",AFL-1.2\nAFL-1.2\nOSL-2.1\nAFL-1.2\nAFL-1.2\nA...,AFL-1.2
14,['4) Exclusions From License Grant. Neither th...,OSL-2.1\nOSL-2.1\nOSL-2.1\nOSL-2.1\nOSL-2.1\nO...,AFL-2.0
15,[],NaN,AFL-2.1\nBSD\nDual-license


In [41]:
def test(
    code_text: str,
    licenses_file_path: str,
    top_k: int = 5,
    min_similarity: int = 50,
    double_semantic_search: bool = False,
):

    licenses = pd.read_csv(licenses_file_path)

    chars_to_remove = ['—', '…', '•', '§', '«', '»', '„', '・', '−', '*', '>', '<']
    for char_to_remove in chars_to_remove:
        code_text = code_text.replace(char_to_remove, '')

    code_chunks = code_text.split('\n')
    temp = []
    for line in code_chunks:
        if line.strip().lower():
            temp.append(line)
    code_chunks = temp

    license_index_map = {}
    all_license_texts = []
    for license_index, license_text in enumerate(licenses['License Text']):
        for line in license_text.split('\n'):
            if line.strip().lower():
                all_license_texts.append(line)
                license_index_map[len(all_license_texts) - 1] = license_index

    extended_results = []
    for start_idx in range(len(code_chunks)):
        code_line = code_chunks[start_idx]
        best_initial_match = None  
        similarity_scores = np.zeros(len(all_license_texts)) 
        for i, lic_line in enumerate(all_license_texts):
            code_line_clean = code_line.strip().lower()        
            lic_line_clean = lic_line.strip().lower()
            if code_line_clean and lic_line_clean:
                similarity_scores[i] = fuzz.ratio(code_line_clean, lic_line_clean)
            else:
                similarity_scores[i] = 0 
        max_index = np.argmax(similarity_scores)
        initial_score = similarity_scores[max_index]
        if initial_score >= min_similarity:
            best_initial_match = (
                initial_score,
                code_line,
                licenses.loc[license_index_map[max_index], 'License Name'],
                licenses.loc[license_index_map[max_index], 'License ID'],
                [
                    (
                        licenses.loc[license_index_map[idx], 'License Name'],
                        similarity_scores[idx]
                    )
                    for idx in np.argsort(similarity_scores[index])[-5:][::-1]
                ],
            )
            def get_n_line_combinations(n, all_lines):
                new_all_lines = []
                for i in range(len(all_lines) - n + 1):
                    current_line = all_lines[i]
                    for j in range(i + 1, i + n):
                        current_line +=f"\n{all_lines[j]}"
                    new_all_lines.append(current_line)
                return new_all_lines

            combined_code_text = code_line
            current_score = initial_score
            for next_idx in range(start_idx + 1, len(code_chunks)):
                combined_code_text += f"\n{code_chunks[next_idx]}"
                lic_line_index = max_index + next_idx - start_idx  
                if lic_line_index >= len(all_license_texts):
                    break  
                current_all_license_texts = get_n_line_combinations(next_idx - start_idx + 1, all_license_texts)
                similarity_scores = np.zeros(len(current_all_license_texts))
                for i, lic_text in enumerate(licenses['License Text']):
                    similarity_scores[i] = fuzz.ratio(combined_code_text, lic_text)

                new_score_index = np.argmax(similarity_scores)
                new_score = similarity_scores[new_score_index]

                if new_score >= (current_score - 5) and new_score >= min_similarity:
                    current_score = new_score
                else:
                    break

            if current_score >= min_similarity:
                # top_5_indices = np.argsort(similarity_scores[index])[-5:][::-1]
                extended_results.append(
                    (
                        current_score,
                        combined_code_text,
                        licenses.loc[license_index_map[lic_line_index], 'License Name'],
                        licenses.loc[license_index_map[lic_line_index], 'License ID'],
                        # [
                        #     (
                        #         licenses.loc[license_index_map[idx], 'License Name'],
                        #         similarity_scores[idx]
                        #     )
                        #     for idx in top_5_indices
                        # ],
                    )
                )
        if best_initial_match is not None:
            extended_results.append(best_initial_match)

    extended_results.sort(key=lambda x: len(x[1]), reverse=True)
    return extended_results

In [ ]:
Okay let's try again.



Here is a list of usable Language elements and their explanations:

"""

AND: When the AND language construct is used between elements of a condition, then the condition only applies, if all elements are fulfilled. When the AND language construct is used between obligations, then all obligation must be fulfilled. Note: The AND language construct is assumed by default between consecutive license obligations and attributes.

ATTRIBUTE: The ATTRIBUTE language construct is used to further specify the term or action from which it depends.

COMPATIBILITY: A license may contain an explicit rule stating that software that is licensed under a particular other license can be combined, copied and distributed along with software that is licensed under the current license. Such other licenses are specified and referenced in this language construct as compatible. Note that a separate statement made anywhere else but not in the license text - even by the authors of the license text - is not accepted here. Exception: Non-copyleft licenses are considered compatible among each other.

COPYLEFT CLAUSE: A license may impose the obligation that any addition or modification of the licensed material must be licensed under the original license. If this is the case, it is noted and referenced under this license construct.

DEPENDING COMPATIBILITY: Another license may contain a rule that the author of the software may explicitly allow to combine, copy and distribute software that is licensed under the other license also under the current license. Since this is not the default case, but requires certain additional paperwork along with the other license, such other license is specified and referenced as to have a depending compatibility.

EITHER IF: The language construct EITHER IF is used to start a list of conditional license obligations. At least one of the listed conditions and the respective obligation must apply.

EITHER: The language construct EITHER is used to start a list of optional license obligations at least one of which must be fulfilled.

EXCEPT IF: A granted right or a license obligation may apply to a large variety of situations, but there may be clearly defined exceptions from such general rules in which case they are encoded using the EXCEPT IF language construct. For better understanding the logic behind this language construct it may be read as "ONLY IF NOT".

IF: The IF language construct is used to make a given term dependent from a condition. In contrast to the USE CASE language construct that denotes a freely selectable option, the IF construct relates to a given situation that either cannot be changed at all or can only be changed with major efforts.

INCOMPATIBILITY: Another license may impose an obligation that the current license does not permit to impose. This creates an incompatibility in such a way that software that is licensed under the other license cannot be included, copied and distributed into software that is licensed under the current license. General rule: Copyleft licenses are incompatible with each other, unless compatibility is explicitly ruled in the license.

INCOMPATIBLE LICENSES: This language construct is only used in merged checklists. It provides a list of licenses that may be incompatible, if the current set of licenses is used to distribute a combined work.

NOT: The NOT language construct is used to negate a subsequent condition.

OR IF: The OR IF language construct describes a conditional license obligation that applies instead of the preceding EITHER IF or another OR IF obligation if the condition is met.

OR: When the OR language construct is used between elements of a condition, then the condition already applies, if only one element is fulfilled. When the OR language construct is used between obligations, then it is sufficient to fulfill at least one obligation. Note: The AND language construct is assumed by default between consecutive license obligations and attributes.

PATENT HINTS: A license may contain a number of hints that refer to the management of possibly or knowingly included patented software components. If this is the case, it is noted and referenced under this license construct.

REMARKS: The language construct REMARKS is used for a particular checklist item to provide additional information that cannot be encoded otherwise.

USE CASE: Sometimes the license obligations may allow the distributor to freely select between a number of optional use cases; the USE CASE language construct is introduced for this purpose. Several USE CASE language constructs to which the same license conditions apply may be combined using the OR language construct. If a particular USE CASE is mentioned repeatedly, e.g. once along with another USE CASE and once not, the obligations of all USE CASE sections must be fulfilled.

YOU MUST NOT: The YOU MUST NOT language construct specifies an individual license prohibition, i.e. what not to do, probably among other things, to become license compliant. It may optionally be followed by indented language constructs such as ATTRIBUTE that further describe the license prohibition.

YOU MUST: The YOU MUST language construct specifies an individual license obligation, i.e. what to do, probably among other things, to become license compliant. It may optionally be followed by indented language constructs such as ATTRIBUTE that further describe the license obligation.

"""



And here is a list of possible actions:

"""

Add

Credit

Delete

Disable

Display

Disseminate

Distribute

Enable

End

Ensure

Exercise

Exhibit

Expire

Forward

Fulfill

Grant

Highlight

Impede

Include

Indemnify

Inform

Litigate

Mark

Misrepresent

Modify

Not do anything else

Notify

Observe

Permit

Prepend

Promote

Provide

Publish

Reference

Rename

Request

Require

Restrict

Search

Sell

Sublicense

Update

Use

"""



And here is a list of terms to use:

"""

Advertisement

Any third party

Appropriate legal notices

Appropriately

Attribution notice

Authorship

Beginning

Binary delivery

Binary-only delivery

Circumstances

Circumvention

Combined library

Combined work delivery

Combined work

Commercial distribution

Compatible license

Contributors

Copyright holder

Copyright license

Copyright notice

Customary medium

Customary method

Database delivery

Debugging

Delayed delivery

Delayed source code delivery

Delivery

Distribution material

Documentation

Documented format

Duration

Dynamic

Effective

Equivalent

Feasible

File

Font delivery

Font

For own use

Functionality

Granted rights

Header file

Human-readable

Identical

Image delivery

Including

Installation information

Installation scripts

Installation

Installed

Interactive

Interface

Interoperable

Irrelevant parts

Legal notices

Liability disclaimer

Library

License acceptance

License announcement

License change

License exception

License fee

License notice

License obligation

License text

License

Linkable work

Linked work

Machine-readable

Modification author

Modification date

Modification notice

Modification reason

Modification report

Modification

Modified library

Modified work

Name

Network service

No charge

No profit

Non-FOSS restrictions

Non-permissive additional terms

Notice

On behalf of

On other server

On same server

Original author

Original license

Original source code

Original work

Other contributors

Other works

Patent holder

Patent license

Patent notice

Preexisting

Prejudicial

Product name

Product

Protocol incompatible

Public domain

Reasonable

Recipient

Relinking

Retrieval information

Reverse engineering

Running

Same medium

Separate

Service offerings

Shared library

Software modification

Source code delivery

Source code modification

Source code

Stand-alone

Standard license notice

Strong copyleft license

Substantial source code modification

Substantial work

Target binary

Technological measures

Third-party attribution notice

Third-party patents

Third-party trademarks

Timely

Title

Tool chain information

Trademark holder

Trademark notice

Transferable

Uncombined

Usage

User product

Verbatim

Via Internet

Via peer-to-peer transmission

Viewable

Warranty disclaimer

Windows code

Work

Written offer

"""



Now write the obligations for the following License Text (Freetype Project License)

"""

Text

The FreeType Project LICENSE



2006-Jan-27



Copyright 1996-2002, 2006 by David Turner, Robert Wilhelm, and Werner Lemberg



Introduction



The FreeType Project is distributed in several archive packages; some of them may contain, in addition to the FreeType font engine, various tools and contributions which rely on, or relate to, the FreeType Project.



This license applies to all files found in such packages, and which do not fall under their own explicit license. The license affects thus the FreeType font engine, the test programs, documentation and makefiles, at the very least.



This license was inspired by the BSD, Artistic, and IJG (Independent JPEG Group) licenses, which all encourage inclusion and use of free software in commercial and freeware products alike. As a consequence, its main points are that:



o We don't promise that this software works. However, we will be interested in any kind of bug reports. (`as is' distribution)

o You can use this software for whatever you want, in parts or full form, without having to pay us. (`royalty-free' usage)

o You may not pretend that you wrote this software. If you use it, or only parts of it, in a program, you must acknowledge somewhere in your documentation that you have used the FreeType code. (`credits')

We specifically permit and encourage the inclusion of this software, with or without modifications, in commercial products. We disclaim all warranties covering The FreeType Project and assume no liability related to The FreeType Project.



Finally, many people asked us for a preferred form for a credit/disclaimer to use in compliance with this license. We thus encourage you to use the following text:



""" Portions of this software are copyright © <year> The FreeType Project (www.freetype.org). All rights reserved. """



Please replace <year> with the value from the FreeType version you actually use.



Legal Terms



0. Definitions

Throughout this license, the terms `package', `FreeType Project', and `FreeType archive' refer to the set of files originally distributed by the authors (David Turner, Robert Wilhelm, and Werner Lemberg) as the `FreeType Project', be they named as alpha, beta or final release.



`You' refers to the licensee, or person using the project, where `using' is a generic term including compiling the project's source code as well as linking it to form a `program' or `executable'. This program is referred to as `a program using the FreeType engine'.



This license applies to all files distributed in the original FreeType Project, including all source code, binaries and documentation, unless otherwise stated in the file in its original, unmodified form as distributed in the original archive. If you are unsure whether or not a particular file is covered by this license, you must contact us to verify this.



The FreeType Project is copyright (C) 1996-2000 by David Turner, Robert Wilhelm, and Werner Lemberg. All rights reserved except as specified below.



1. No Warranty

THE FREETYPE PROJECT IS PROVIDED `AS IS' WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING, BUT NOT LIMITED TO, WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE. IN NO EVENT WILL ANY OF THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY DAMAGES CAUSED BY THE USE OR THE INABILITY TO USE, OF THE FREETYPE PROJECT.



2. Redistribution

This license grants a worldwide, royalty-free, perpetual and irrevocable right and license to use, execute, perform, compile, display, copy, create derivative works of, distribute and sublicense the FreeType Project (in both source and object code forms) and derivative works thereof for any purpose; and to authorize others to exercise some or all of the rights granted herein, subject to the following conditions:



o Redistribution of source code must retain this license file (`FTL.TXT') unaltered; any additions, deletions or changes to the original files must be clearly indicated in accompanying documentation. The copyright notices of the unaltered, original files must be preserved in all copies of source files.

o Redistribution in binary form must provide a disclaimer that states that the software is based in part of the work of the FreeType Team, in the distribution documentation. We also encourage you to put an URL to the FreeType web page in your documentation, though this isn't mandatory.

These conditions apply to any software derived from or based on the FreeType Project, not just the unmodified files. If you use our work, you must acknowledge us. However, no fee need be paid to us.



3. Advertising

Neither the FreeType authors and contributors nor you shall use the name of the other for commercial, advertising, or promotional purposes without specific prior written permission.



We suggest, but do not require, that you use one or more of the following phrases to refer to this software in your documentation or advertising materials: `FreeType Project', `FreeType Engine', `FreeType library', or `FreeType Distribution'.



As you have not signed this license, you are not required to accept it. However, as the FreeType Project is copyrighted material, only this license, or another one contracted with the authors, grants you the right to use, distribute, and modify it. Therefore, by using, distributing, or modifying the FreeType Project, you indicate that you understand and accept all the terms of this license.



4. Contacts

There are two mailing lists related to FreeType:



o freetype@nongnu.org

Discusses general use and applications of FreeType, as well as future and wanted additions to the library and distribution. If you are looking for support, start in this list if you haven't found anything to help you in the documentation.



o freetype-devel@nongnu.org

Discusses bugs, as well as engine internals, design issues, specific licenses, porting, etc.



Our home page can be found at



http://www.freetype.org



--- end of FTL.TXT ---

"""